In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import zipfile  # noqa: E402
import numpy as np  # noqa: E402
import numpy.lib.format as npformat  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402
from scipy.stats import zscore  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.analysis.pca_polarity import (  # noqa: E402
    align_pc1_signs,
    apply_pc1_signs,
    topography_consistency,
)
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    MusicTypeVariants,
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# ASSR Raw Voltage — Channel-PCA Components (topomap + time course)

Raw-voltage analog of `assr_wavelet_pca_analysis.ipynb`. Instead of wavelet power,
this reduces the stimulus-locked **evoked voltage** to a small number of **spatial
components per participant** by collapsing the channel dimension with a **PCA over
channels**.

The first `N_COMPONENTS` components (default **3**) are extracted and treated
**identically** — same z-scoring, same trial average, same cross-participant
polarity alignment, same diagnostics, same plots — so PC2/PC3 can be compared
against PC1 rather than assumed to be noise. The dominant-variance component is not
necessarily the best ASSR carrier: PC1 captures whatever spatial mode explains the
most channel variance in the evoked window, which may be a slow onset response,
while the 40 Hz steady-state can live in a later component. The final comparison
cell scores every component on the same measures.

Pipeline (per subject, on the **Placebo** condition by project convention):

1. **Trial average.** Epoch the raw signal around every `fam+` onset and average
   over stimuli — the phase-locked *evoked* response `(n_channels, win)`. The epoch
   is **0.1 s before onset to 1.0 s after** it — the 0.5 s stimulus plus 0.5 s of
   post-stimulus — from `src.definitions.constants.AssrEpoch`, capped by the
   shortest inter-onset gap so no epoch touches a neighbouring stimulus.
2. **Z-score per channel** (over the whole recording) so every electrode has unit
   variance and contributes **equally** to the PCA. Without this the channel PCA
   is a covariance-PCA dominated by the highest-variance electrodes; z-scoring
   turns it into a correlation-PCA with equal electrode influence. The trade-off:
   the component score is then in normalized units, not µV.
3. **Channel PCA.** Treat each **time sample** as an observation and each channel
   as a variable — matrix `X` of shape `(win, n_channels)` — and fit PCA. Each
   retained component gives two things: its **loading vector** `(n_channels,)` is
   the component's scalp **topography**, and its **score** `(win,)` is the
   component's **time course**.
4. **Polarity alignment across participants**, done **per component and
   independently** (each component's sign is its own arbitrary choice). Every
   subject's loading is aligned to a common template (iteratively-refined group
   mean) and the overall orientation anchored to a reference channel
   (`src.analysis.pca_polarity`), which maximises cross-subject topography
   agreement; the score is flipped with the loading so topography and time course
   stay consistent.
5. **Plot** the per-participant topomap and time course, one figure set per
   component, then a **cross-component comparison** (explained variance,
   topography consistency, 40 Hz steady-state SNR).

Z-scoring is per channel against the **whole recording** (not per epoch), matching
the wavelet notebook's normalization rationale. The stimulus alignment (see
`00-preprocessing/stimulus_alignment.ipynb`) puts every onset at the same sample
index in all participants, so one shared onsets array serves every subject.

> **Component identity is per subject, not global.** Each participant gets their
> own PCA, so "PC2" means "that subject's second-most-variance mode" — subjects can
> disagree about what that mode is, and there is no guarantee PC2 means the same
> thing everywhere. The topography-consistency diagnostic is the check: PC1
> typically agrees across subjects, and a later component whose consistency
> collapses is not describing one shared mode, so its group average is not
> interpretable no matter how good its per-subject waveforms look.

> Resolving the sign matters more than it looks: averaging arbitrarily signed
> loading maps shrinks the group mean toward `1/sqrt(n_subjects)` of the aligned
> amplitude and leaves noise behind. On this dataset, unresolved signs cost ~13% of
> the PC1 group waveform's split-half reliability and ~22% of its amplitude. The
> polarity cell prints a **consistency diagnostic per component** — read it before
> trusting any group-average panel.

> The raw concatenated array is ~1 GB and read **memory-mapped**, so every subject
> is cheap — no lazy streaming is needed (contrast the 49 GB wavelet cache). The
> wavelet file is opened only to read its small channel-name member.

## Configuration

In [ ]:
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO   # default per project convention
MUSIC_TYPE = MusicTypeVariants.ASSR

# ── Subjects to reduce ────────────────────────────────────────────────────
# None = ALL subjects (the raw array is memory-mapped, so every subject is cheap).
# Set to an explicit list of CONCATENATED_PERSON_INDEX values to subset.
SUBJECT_INDICES = None

# ── Components to extract ─────────────────────────────────────────────────
# Number of leading channel-PCA components kept per participant. All of them are
# processed IDENTICALLY (polarity alignment, diagnostics, plots), so a later
# component can be compared against PC1 instead of being assumed uninformative —
# the highest-variance mode is not necessarily the best 40 Hz carrier. Set to 1 to
# recover the original PC1-only behaviour.
N_COMPONENTS = 3

# ── Stimulus-locked epoch window ──────────────────────────────────────────
# Taken from the paradigm definition (src.definitions.constants.AssrEpoch) so
# every onset-locked ASSR analysis cuts the same window:
#   PRE_PAD_S  = 0.1 s baseline before onset
#   POST_PAD_S = 1.0 s after onset = 0.5 s stimulus + 0.5 s post-stimulus
# The post length is capped by the shortest inter-onset gap in the subset cell, so
# no epoch can reach a neighbouring stimulus.
PRE_PAD_S = AssrEpoch.PRE_ONSET_S
POST_PAD_S = AssrEpoch.POST_ONSET_S

ASSR_FREQ = 40.0           # expected steady-state frequency (Hz)
SFREQ = 250.0              # sampling rate of the concatenated data

# Z-score each channel against the WHOLE recording before epoching, so every
# electrode has unit variance and contributes equally to the channel PCA
# (correlation-PCA). Set False to run PCA on raw µV (covariance-PCA, dominated by
# high-variance electrodes; the time course is then in µV).
ZSCORE_PER_CHANNEL = True

# ── Channel-name source ───────────────────────────────────────────────────
# The concatenated channel order is recovered from the wavelet cache's small
# `feature_names` member (only that member is read — the 49 GB data stream is
# never touched). Needed to map component loadings onto the topomap montage order.
WAVELET_FREQ_SIG = "1.000_50.000_50"   # freqs[0]_freqs[-1]_n_freqs
N_WAVELET_FREQS = 50

# ── Plot saving ───────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "assr_raw_pca"
    / f"{CONDITION.value}_{MUSIC_TYPE.value}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Group          : {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Subjects       : {'ALL' if SUBJECT_INDICES is None else SUBJECT_INDICES}")
print(f"Components     : {N_COMPONENTS} (PC1..PC{N_COMPONENTS})")
print(f"Pre-onset pad  : {PRE_PAD_S} s")
print(f"Post-onset span: {POST_PAD_S} s "
      f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
      f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)")
print(f"Z-score/channel: {ZSCORE_PER_CHANNEL}")
print(f"Plots -> {PLOTS_DIR}")

## Load Concatenated Raw Data, Onsets, Metadata & Channel Names

In [ ]:
safe_label = f"{CONDITION.value}_{MUSIC_TYPE.value}"

concat_dir = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / PreprocessedDataVariants.CONCATENATED.value
)
raw_path = concat_dir / f"{safe_label}.npy"
onsets_path = concat_dir / f"{safe_label}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
meta_path = concat_dir / f"{safe_label}.metadata.csv"

wavelet_path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / "wavelets"
    / "broadband"
    / f"{safe_label}__wavelet_power__{WAVELET_FREQ_SIG}__freqdim1.npz"
)
for p in (raw_path, onsets_path, meta_path, wavelet_path):
    assert p.exists(), f"Missing expected file: {p}"

# Memory-mapped raw array (~1 GB) — only the accessed subjects land in memory.
raw_mm = np.load(raw_path, mmap_mode="r")            # (n_subj, n_ch, n_times)
onsets = np.load(onsets_path)                        # (n_onsets,) shared sample idx
meta = pd.read_csv(meta_path, index_col=0)
n_subj_total, n_channels, n_times_raw = raw_mm.shape

# Channel names live in the wavelet feature_names (layout: channel*N_FREQS). Only
# that small member is decompressed; the concatenated raw shares the same order.
with zipfile.ZipFile(wavelet_path) as _z:
    with _z.open("feature_names.npy") as _f:
        feature_names = npformat.read_array(_f, allow_pickle=True)
channel_names = [str(feature_names[i * N_WAVELET_FREQS]).split("@")[0]
                 for i in range(len(feature_names) // N_WAVELET_FREQS)]
assert len(channel_names) == n_channels, (
    f"channel-name count {len(channel_names)} != raw n_channels {n_channels}"
)

gaps = np.diff(onsets)
print(f"Concatenated raw : {raw_mm.shape}  (dtype {raw_mm.dtype})")
print(f"Stimulus onsets  : {onsets.shape}  range [{onsets.min()}, {onsets.max()}]")
print(f"Inter-onset gap  : min {int(gaps.min())}, median {int(np.median(gaps))}, "
      f"max {int(gaps.max())} samples")
print(f"Channels parsed  : {n_channels} (first={channel_names[0]}, "
      f"last={channel_names[-1]})")
meta[["SingleDataMetadata.PARTICIPANT_ID", "SingleDataMetadata.CONDITION",
      "SingleDataMetadata.CONCATENATED_PERSON_INDEX"]].head()

## Subset Selection & Epoch Window

The epoch comes from the **paradigm**, not from the observed jitter: `PRE_PAD_S` = 0.1 s
of baseline before onset, then `POST_PAD_S` = 1.0 s after onset — the **0.5 s stimulus
plus 0.5 s post-stimulus**, so a response outlasting the stimulus stays visible. Both
values live in `src.definitions.constants.AssrEpoch`, shared with the wavelet notebook
and the onset-locked IVA quality references.

The post-onset length is then **capped** by the shortest inter-onset gap, so no epoch can
reach a neighbouring stimulus. With ~1.25 s between onsets the full 1.1 s epoch fits
comfortably inside one inter-onset interval — including the pre-onset baseline, which the
earlier `gaps.min()`-derived window did not (it made the "baseline" the tail of the
previous stimulus).

`stim_mask` marks the driven interval. Use it for any steady-state measure: a 1 s epoch
around a 0.5 s stimulus is half silence, which dilutes an estimate computed over the
whole post-onset window.

In [ ]:
subject_indices = (
    list(range(n_subj_total)) if SUBJECT_INDICES is None else list(SUBJECT_INDICES)
)

# Epoch window in samples: paradigm span, capped by the shortest inter-onset gap.
PRE = int(round(PRE_PAD_S * SFREQ))
POST = min(int(round(POST_PAD_S * SFREQ)), int(gaps.min()))
epoch_times = np.arange(-PRE, POST) / SFREQ         # (win,) seconds, t=0 at onset
stim_mask = AssrEpoch.stimulus_mask(epoch_times)    # driven interval, for later use

# Map subject indices -> participant labels via the concatenated metadata.
pidx_col = "SingleDataMetadata.CONCATENATED_PERSON_INDEX"
pid_col = "SingleDataMetadata.PARTICIPANT_ID"
idx_to_pid = dict(zip(meta[pidx_col], meta[pid_col].astype(str).str.zfill(3)))
# Order subjects by participant ID so every per-participant plot is sorted by PID
# rather than by concatenated index.
subject_indices = sorted(
    subject_indices, key=lambda si: int(idx_to_pid.get(si, "9999"))
)
subject_labels = [f"PSI{idx_to_pid.get(si, '???')}" for si in subject_indices]

print(f"Selected subjects: {dict(zip(subject_indices, subject_labels))}")
print(f"Epoch window     : {PRE + POST} samples ({PRE} pre, {POST} post) "
      f"= [{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")
print(f"Stimulus interval: [0.000, {AssrEpoch.STIMULUS_DURATION_S:.3f}] s "
      f"({int(stim_mask.sum())} samples); post-stimulus "
      f"{epoch_times[-1] - AssrEpoch.STIMULUS_DURATION_S:.3f} s")
if POST < int(round(POST_PAD_S * SFREQ)):
    print(f"  NOTE: post-onset span trimmed from {POST_PAD_S} s to "
          f"{POST / SFREQ:.3f} s by the shortest inter-onset gap "
          f"({int(gaps.min())} samples).")
if PRE + POST > int(gaps.min()):
    print(f"  WARNING: epoch ({PRE + POST} samples) exceeds the shortest gap "
          f"({int(gaps.min())}) — the baseline reaches into the previous stimulus.")

## Helper — stimulus-locked trial averaging

In [ ]:
def epoch_average(arr, onsets, pre, post):
    """Average fixed windows around each onset along the LAST axis.

    Args:
        arr: array whose last axis is time, e.g. ``(n_channels, n_times)``.
        onsets: stimulus onset sample indices.
        pre, post: samples kept before / after each onset (window = pre + post).

    Returns:
        ``(averaged, n_used)`` where the time axis is replaced by the
        ``pre + post`` window, averaged over every onset whose window fits inside
        the recording (edge windows are skipped).
    """
    n_time = arr.shape[-1]
    acc = None
    n_used = 0
    for o in onsets:
        s, e = o - pre, o + post
        if s < 0 or e > n_time:
            continue
        seg = arr[..., s:e]
        acc = seg.astype(np.float64) if acc is None else acc + seg
        n_used += 1
    if n_used == 0:
        raise ValueError("No onset window fits inside the recording.")
    return acc / n_used, n_used


print("Helper defined.")

## Trial-Average & Channel PCA — leading components per participant

For each subject: optionally z-score every channel against the whole recording
(equal electrode influence), epoch the signed signal around every onset and average
(the evoked response, `(n_channels, win)`), then fit a PCA where each **time
sample** is an observation and each **channel** a variable. Each of the first
`N_COMPONENTS` components gives a **score** — its time course `(win,)` — and a
**loading** — its scalp topography `(n_channels,)`.

Every component goes through the **exact same** treatment; nothing downstream is
special-cased for PC1. The point is comparability: PCA orders components by
explained *channel variance* in the evoked window, which is not the same thing as
carrying the 40 Hz steady-state. A large slow onset deflection can dominate PC1 and
push the ASSR into PC2 or PC3, so the later components have to be scored on the
same measures before they can be dismissed.

**Polarity alignment across participants, per component.** A PCA component's sign is
arbitrary, so some participants come out with flipped polarity — and each component
is flipped independently of the others, so alignment runs **separately per
component**. Every subject's loading is aligned to a common **template** (the
iteratively-refined group-mean loading), which **maximises cross-subject topography
agreement** — the criterion that matters here, because the point is to find what
participants share and aggregate it. A subject whose topography strongly
*anti*-correlates with the others is almost always just sign-flipped, and template
alignment is what un-flips it. The overall orientation is then anchored to a
reference channel (`Cz` if present) so the result is reproducible, and the score is
flipped with the loading so each subject's topography and time course stay
consistent.

> A purely per-participant criterion (e.g. "flip when the loading sums negative")
> cannot do this job: these evoked loadings are **dipolar**, so their entries nearly
> cancel (`|sum| / |loading|₁` runs 0.03–0.19 on this dataset, against 1.0 for a
> uniformly signed map). The sign would then be decided by a few percent of residual
> imbalance, and subjects come out mutually inverted despite each one individually
> "summing positive".

The cell prints a **consistency diagnostic per component**: after alignment every
subject should correlate positively with the group-mean topography. A subject that
still does not is a genuine topographic outlier rather than a sign problem, and
averaging it into the group map cancels real signal. Expect consistency to fall off
for later components — that is itself the finding, because a component that no two
subjects share has no meaningful group average. Sign alignment cannot rescue a
component that subjects genuinely disagree about; it can only remove the sign
ambiguity, so weak consistency here means the group panels for that component
should not be read as a group effect.

In [ ]:
subs = list(subject_indices)
n_subj = len(subs)
win = PRE + POST
pc_labels = [f"PC{c + 1}" for c in range(N_COMPONENTS)]

# Component-major storage: axis 0 = component, axis 1 = subject (in the order of
# `subs` / `subject_labels`). Every component is filled by the SAME code path.
scores = np.empty((N_COMPONENTS, n_subj, win))          # time courses
loadings = np.empty((N_COMPONENTS, n_subj, n_channels))  # spatial patterns
explained = np.empty((N_COMPONENTS, n_subj))             # explained variance ratio

n_used = None
for k, si in enumerate(subs):
    sig = np.asarray(raw_mm[si])                    # (n_ch, n_times)
    if ZSCORE_PER_CHANNEL:
        # Unit variance per channel over the WHOLE recording -> every electrode
        # contributes equally to the channel PCA (correlation-PCA).
        sig = zscore(sig, axis=1)
    evoked, n_used = epoch_average(sig, onsets, PRE, POST)  # (n_ch, win)
    # PCA over channels: observations = time samples, variables = channels.
    X = evoked.T                                    # (win, n_channels)
    pca = PCA(n_components=N_COMPONENTS)
    scores[:, k, :] = pca.fit_transform(X).T        # (n_pc, win) time courses
    loadings[:, k, :] = pca.components_             # (n_pc, n_channels) loadings
    explained[:, k] = pca.explained_variance_ratio_

# ── Align polarity ACROSS participants, INDEPENDENTLY PER COMPONENT ───────
# A PCA component's sign is arbitrary, so some participants come out flipped — and
# each component is flipped independently of the others, so each gets its own
# alignment pass. Every subject's loading is flipped toward a common template (the
# iteratively-refined group-mean loading), which MAXIMISES cross-subject topography
# agreement — the criterion that matters when the goal is to aggregate
# participants. A subject strongly anti-correlated with the others is almost always
# just sign-flipped, and this un-flips it. The overall orientation is then anchored
# to a reference channel so the result is reproducible. The same sign is applied to
# the loading (topomap) AND the score (time course) so each subject stays
# internally consistent.
consistency = []   # per component: SignConsistency of the aligned loadings
n_flipped = []     # per component: how many subjects were sign-flipped
anchors = []       # per component: channel that fixed the overall orientation
for c in range(N_COMPONENTS):
    signs, anchor = align_pc1_signs(loadings[c], channel_names=channel_names)
    loadings[c] = apply_pc1_signs(loadings[c], signs)
    scores[c] = apply_pc1_signs(scores[c], signs)
    # Diagnostic: after alignment every subject should correlate POSITIVELY with
    # the group-mean topography. One that does not is a topographic outlier, not a
    # sign problem — averaging it into the group map cancels real signal.
    consistency.append(topography_consistency(loadings[c]))
    n_flipped.append(int((signs < 0).sum()))
    anchors.append(anchor)

print(f"Trial-averaged {n_used} stimuli per participant; PCA over {n_channels} "
      f"channels, {N_COMPONENTS} component(s) kept, {n_subj} participants.")
for c, pc in enumerate(pc_labels):
    cons = consistency[c]
    print(f"\n{pc}  (mean explained variance {explained[c].mean() * 100:.1f}%, "
          f"range {explained[c].min() * 100:.1f}–{explained[c].max() * 100:.1f}%)")
    print(f"  polarity aligned      : {n_flipped[c]} subject(s) flipped; "
          f"anchor {anchors[c]}")
    print(f"  topographies agreeing : {cons.n_agreeing}/{cons.n_subjects}")
    print(f"  median pairwise r     : {cons.median_pairwise_r:+.3f}")
    print(f"  weakest subject r     : {cons.min_subject_r:+.3f}")
    if cons.n_agreeing < cons.n_subjects:
        print(f"  WARNING: {cons.n_subjects - cons.n_agreeing} subject(s) still "
              f"anti-correlate with the group topography — inspect before "
              f"trusting the {pc} group panels.")
    print("  per-subject explained variance: " + ", ".join(
        f"{label} {explained[c, k] * 100:.1f}%"
        for k, label in enumerate(subject_labels)
    ))

### Per-Participant Time Course — one figure per component

One grid per component, all drawn with the same code and the same axes, so PC1,
PC2 and PC3 are read the same way. A 40 Hz steady-state shows up as a visible
ripple through the driven interval (shaded); a slow onset deflection does not.

In [ ]:
unit = "z-scored" if ZSCORE_PER_CHANNEL else "µV"
ncols = min(5, n_subj)
nrows = int(np.ceil(n_subj / ncols))
stim_end = AssrEpoch.STIMULUS_DURATION_S

for c, pc in enumerate(pc_labels):
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(3.6 * ncols, 2.7 * nrows), sharex=True, squeeze=False
    )
    flat = axes.flatten()
    for ax, k in zip(flat, range(n_subj)):
        ax.plot(epoch_times, scores[c, k], lw=1.5, color=f"C{c}")
        ax.axvspan(0.0, stim_end, color="grey", alpha=0.12, lw=0)
        ax.axvline(0.0, color="red", ls="--", lw=0.8)
        ax.set_title(f"{subject_labels[k]}  ({pc} {explained[c, k] * 100:.0f}%)",
                     fontsize=9)
    for ax in flat[n_subj:]:
        ax.axis("off")
    fig.supxlabel("Time relative to onset (s)")
    fig.supylabel(f"{pc} projected signal ({unit}, polarity-aligned)")
    fig.suptitle(
        f"Per-participant channel-PCA {pc} time course — "
        f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj}, "
        f"mean EV {explained[c].mean() * 100:.0f}%)",
        y=1.0,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / f"raw_pca_{pc.lower()}_timecourse_per_participant.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

### Time Course — overlay + group mean, one row per component

All participants overlaid with the across-participant mean, stacked one row per
component on a shared time axis. Polarity is aligned across participants within
each component (loadings matched to a common template), so each mean is a
meaningful summary wherever that component's spatial mode is shared across subjects
— check the per-component consistency diagnostic printed above before reading a row
whose topographies disagree.

Each row keeps its own y-scale: component scores shrink with component order by
construction, so a shared scale would flatten PC2/PC3 into lines and hide exactly
the waveform shape being compared. Amplitudes are therefore **not** comparable
between rows; the explained-variance and SNR numbers in the comparison cell are.

In [ ]:
unit = "z-scored" if ZSCORE_PER_CHANNEL else "µV"
group_tc = scores.mean(axis=1)          # (n_pc, win) across-participant mean

fig, axes = plt.subplots(
    N_COMPONENTS, 1, figsize=(11, 4.2 * N_COMPONENTS), sharex=True, squeeze=False
)
for c, pc in enumerate(pc_labels):
    ax = axes[c, 0]
    for k, label in enumerate(subject_labels):
        ax.plot(epoch_times, scores[c, k], lw=1.1, alpha=0.75, label=label)
    ax.plot(epoch_times, group_tc[c], lw=2.6, color="black", label="group mean")
    ax.axvspan(0.0, stim_end, color="grey", alpha=0.12, lw=0)
    ax.axvline(0.0, color="red", ls="--", lw=1, label="onset")
    ax.set_title(
        f"{pc} — mean EV {explained[c].mean() * 100:.0f}%, "
        f"topographies agreeing {consistency[c].n_agreeing}/"
        f"{consistency[c].n_subjects}, median pairwise r "
        f"{consistency[c].median_pairwise_r:+.2f}",
        fontsize=10,
    )
    ax.set_ylabel(f"{pc} ({unit})")
    if c == 0:
        ax.legend(loc="upper right", fontsize=7, ncol=3)
axes[-1, 0].set_xlabel("Time relative to onset (s)")
fig.suptitle(
    f"Channel-PCA component time courses — "
    f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj})",
    y=1.0,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "raw_pca_timecourse_overlay.png", dpi=150,
                bbox_inches="tight")
plt.show()

## Topomap Montage — electrode positions

Electrode positions come from one `RAW_CROPPED` recording's `Info` (same channel
order guaranteed by remapping onto the canonical `channel_names`).

In [ ]:
topo_handler = DatasetHandler(EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS)
info_fname = meta.loc[
    meta[pidx_col] == subject_indices[0], "SingleDataMetadata.FILENAME"
].iloc[0]
topo_info = (
    topo_handler.load_data_file(
        info_fname,
        is_processed=True,
        processed_data_type=PreprocessedDataVariants.RAW_CROPPED,
        preload=False,
    )
    .pick("eeg")
    .info
)
name_pos = {n: i for i, n in enumerate(channel_names)}
assert set(topo_info["ch_names"]) <= set(name_pos), (
    "RAW_CROPPED channels are not a subset of the concatenated channel names."
)
info_order = [name_pos[n] for n in topo_info["ch_names"]]
print(f"Topomap montage: {len(topo_info['ch_names'])} electrodes "
      f"(reordered onto concatenated channel order).")

### Per-Participant Topomap — one figure per component

Scalp topography of each component per participant (loading vector), following the
notebook-05 convention — signed values, diverging `RdBu_r`, symmetric colour scale,
group average as the last panel. Every component is plotted by the same code with
its own colour scale (loadings are unit-norm, so scales are already comparable in
magnitude).

Polarity is aligned across participants within each component, so **all panels of a
figure should share the same sign structure** — that is exactly what the alignment
enforces — and the group-average panel is meaningful. A panel that still looks
inverted relative to the rest was flagged by that component's consistency
diagnostic as a topographic outlier. Where the diagnostic is weak, expect the group
panel to look washed out: that is subjects' maps cancelling, not a weak effect.

In [ ]:
group_loading = loadings.mean(axis=1)          # (n_pc, n_channels)

for c, pc in enumerate(pc_labels):
    panels = [
        (f"{subject_labels[k]}  ({pc} {explained[c, k] * 100:.0f}%)",
         loadings[c, k][info_order])
        for k in range(n_subj)
    ]
    panels.append((f"group average  (mean {pc} {explained[c].mean() * 100:.0f}%)",
                   group_loading[c][info_order]))

    # Symmetric shared scale (nb05 convention): 99th percentile of |loading|.
    vlim = float(np.percentile(np.abs(np.concatenate([v for _, v in panels])), 99))
    if vlim == 0.0:
        vlim = 1e-12

    n_panels = len(panels)
    ncols = min(5, n_panels)
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.7 * ncols, 2.9 * nrows),
                             squeeze=False)
    flat = axes.flatten()
    im = None
    for ax, (label, vals) in zip(flat, panels):
        im, _ = mne.viz.plot_topomap(
            vals, topo_info, axes=ax, show=False, cmap="RdBu_r",
            vlim=(-vlim, vlim), contours=4,
        )
        ax.set_title(label, fontsize=9)
    for ax in flat[n_panels:]:
        ax.axis("off")
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6,
                 label=f"{pc} loading (a.u.)")
    fig.suptitle(
        f"Per-participant channel-PCA {pc} topography — "
        f"{CONDITION.value}/{MUSIC_TYPE.value} (n={n_subj}, "
        f"agreeing {consistency[c].n_agreeing}/{consistency[c].n_subjects})",
        y=1.0,
    )
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / f"raw_pca_{pc.lower()}_topomap_per_participant.png",
                    dpi=150, bbox_inches="tight")
    plt.show()

## Component Comparison — is PC1 actually the best one?

Every component was extracted, aligned and plotted identically, so they can be
scored side by side. PCA ranks components by explained **channel variance** in the
evoked window, which is not the quantity of interest here — a large slow onset
deflection can own PC1 while the 40 Hz steady-state sits in PC2 or PC3. The table
below reports, per component:

- **mean EV** — mean explained variance ratio across participants (what PCA
  actually maximised).
- **agreeing / median r / min r** — the topography-consistency diagnostic on the
  aligned loadings. This is the gate: a component whose subjects disagree has no
  interpretable group average, however clean its individual waveforms look.
- **40 Hz SNR (subject)** — median across participants of the power at `ASSR_FREQ`
  relative to neighbouring spectral bins, computed on each subject's own score
  within the **driven interval only** (a 1.1 s epoch around a 0.5 s stimulus is
  half silence, which dilutes a steady-state estimate).
- **40 Hz SNR (group mean)** — the same measure on the across-participant mean
  waveform. Higher than the per-subject median means the response is
  **phase-consistent across participants** and survives averaging; much lower means
  subjects' 40 Hz is present but not aligned in phase, so the group mean cancels it.
- **n SNR > 3** — how many participants individually clear an SNR of 3.

The stimulus window is 0.5 s, so the spectral resolution is 2 Hz and 40 Hz falls on
an exact bin. A Hann taper spreads the peak into its immediate neighbours, so those
are excluded from the noise floor.

Read the table this way: a later component is the better ASSR carrier only if it
beats PC1 on the 40 Hz measures **and** holds up on topography consistency. Winning
on SNR while failing consistency means individual subjects have 40 Hz in some
second mode, but not in the *same* second mode — worth following up per subject,
not worth group-averaging.

In [ ]:
SNR_THRESHOLD = 3.0        # per-subject 40 Hz SNR counted as a hit
N_SIDE_BINS = 3            # noise-floor bins on each side of the ASSR bin
N_EXCLUDE_BINS = 1         # bins next to the peak skipped (Hann taper leakage)


def stimulus_spectrum(x):
    """Hann-tapered power spectrum of the driven interval of one score.

    Args:
        x: ``(win,)`` component time course over the full epoch.

    Returns:
        ``(freqs, power)`` for the stimulus interval only, mean-removed.
    """
    seg = np.asarray(x, dtype=float)[stim_mask]
    seg = (seg - seg.mean()) * np.hanning(seg.size)
    power = np.abs(np.fft.rfft(seg)) ** 2
    freqs = np.fft.rfftfreq(seg.size, 1.0 / SFREQ)
    return freqs, power


def assr_snr(x, freq=ASSR_FREQ):
    """Power at *freq* relative to the median of neighbouring bins."""
    freqs, power = stimulus_spectrum(x)
    peak = int(np.argmin(np.abs(freqs - freq)))
    lo_hi = peak - N_EXCLUDE_BINS
    side = np.concatenate([
        power[max(lo_hi - N_SIDE_BINS, 0):max(lo_hi, 0)],
        power[peak + N_EXCLUDE_BINS + 1:peak + N_EXCLUDE_BINS + 1 + N_SIDE_BINS],
    ])
    floor = float(np.median(side)) if side.size else float("nan")
    return float(power[peak] / floor) if floor > 0 else float("nan")


subject_snr = np.array([[assr_snr(scores[c, k]) for k in range(n_subj)]
                        for c in range(N_COMPONENTS)])   # (n_pc, n_subj)
group_snr = np.array([assr_snr(group_tc[c]) for c in range(N_COMPONENTS)])

comparison = pd.DataFrame({
    "component": pc_labels,
    "mean_EV_%": [explained[c].mean() * 100 for c in range(N_COMPONENTS)],
    "agreeing": [f"{consistency[c].n_agreeing}/{consistency[c].n_subjects}"
                 for c in range(N_COMPONENTS)],
    "median_pairwise_r": [consistency[c].median_pairwise_r
                          for c in range(N_COMPONENTS)],
    "min_subject_r": [consistency[c].min_subject_r for c in range(N_COMPONENTS)],
    f"{ASSR_FREQ:.0f}Hz_SNR_subject_median": np.nanmedian(subject_snr, axis=1),
    f"{ASSR_FREQ:.0f}Hz_SNR_group_mean": group_snr,
    f"n_SNR>{SNR_THRESHOLD:.0f}": (subject_snr > SNR_THRESHOLD).sum(axis=1),
}).set_index("component")

best_snr = pc_labels[int(np.nanargmax(group_snr))]
best_cons = pc_labels[int(np.argmax([c.median_pairwise_r for c in consistency]))]
print(f"Driven interval: {int(stim_mask.sum())} samples "
      f"({stim_mask.sum() / SFREQ:.3f} s) -> {SFREQ / stim_mask.sum():.1f} Hz "
      f"spectral resolution.")
print(f"Highest {ASSR_FREQ:.0f} Hz SNR on the group mean : {best_snr}")
print(f"Most consistent topography across subjects: {best_cons}")
comparison.round(3)